# 2.3 Multi-agent (subagents as tools)

A **supervisor** pattern: the main agent does not own the math tools. It calls **subagents**, each with a single tool.

Each subagent is itself `create_agent`. The main agent only sees wrapper tools (`call_subagent_1`, `call_subagent_2`) that `invoke` those graphs and return the last message.

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Creating subagents

Two specialists: square root vs square. Isolating tools keeps each subagent's prompt and tool list small.

In [ ]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [ ]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[square]
)

## Calling subagents

Wrapper tools take `x`, invoke the matching subagent with a `HumanMessage`, and return `response["messages"][-1].content` to the supervisor.

The main agent's system prompt says **when** to call which wrapper.

In [ ]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

Ask for a square root. The outer trace should show `call_subagent_1`; inside that call, `square_root` runs.

In [ ]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [ ]:
from pprint import pprint

pprint(response)